# Act 4 — A planning agent and its harness

These are the lesson cells tested in Colab. Run this notebook from top to bottom; it also works independently of the other act notebooks.

**Setup:** use a Python 3.13+ kernel. The next cells find or clone the repository and install its pinned direct dependencies. If your Colab setup already did this, skip those two cells and run the imports and settings below from the repository directory.

All imports are grouped here, above the lessons. Set any model environment override before running the lesson imports.


In [ ]:
from pathlib import Path
from getpass import getpass
import tomllib
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

In [ ]:
import os
from getpass import getpass
from pathlib import Path

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import build_act2, preview_context
from acts.act3_workflow import build_act3
from acts.act4_harness import build_act4
from acts.act5_skills import build_act5
from acts.act6_composition import build_act6
from cli import (
    print_agent_result,
    print_cart,
    print_workflow_result,
    print_harness_result,
    print_composition_result,
)
from formaggio.config import MODEL
from formaggio.agents.context import (
    customer_ask, customer_message, instructions, load_scenario,
)
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store
from formaggio.operations.chat_view import show_chat


## Before we begin

We will build a cheese-shop assistant in six steps. The shop data and rules are fictional classroom examples.

The cells use live model calls by default. Set `USE_FIXTURES = True` to use scripted responses in Acts 3–6. Acts 1–2 still use the live model.

When a review is needed, you will see the cart or email draft and type `approve` or `decline`. Orders are simulated, and approved emails are saved as local HTML files; no purchases or emails are sent.


In [ ]:
SCENARIO = "standard"  # Keep this the same for Acts 1–2.
USE_FIXTURES = False

DB_PATH = Path("outputs/colab.sqlite")
OUTPUT_ROOT = Path("outputs/colab_artifacts")
MEMORY_PATH = Path("outputs/colab_memory.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def ask_for_decision():
    while True:
        decision = input("Type approve or decline: ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision
        print("Please enter approve or decline.")


async def review_order(ticket):
    print("\n👤 Manager review")
    print(customer_ask(ticket.request))
    print_cart(ticket.report)
    return ask_for_decision()


async def review_email(review):
    print("\n👤 Review this email draft")
    print(f"To: {review.draft.recipient}")
    print(f"Subject: {review.draft.subject}\n")
    print(review.draft.body)
    return ask_for_decision()


print(f"Model: {MODEL}")
print("Acts 3–6:", "scripted responses" if USE_FIXTURES else "live model")


### API key

If you already configured `OPENAI_API_KEY` in Colab, this cell reuses it. Otherwise it asks privately when live calls are enabled. The key is not printed or saved in the notebook.


In [ ]:
if not USE_FIXTURES and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


## Act 4 — Help an agent manage a longer task

This act introduces a planning agent. It checks an event menu, investigates a stock shortage, and prepares a supplier email for your review.

Its harness adds a task list, saved customer preferences, and a way to shorten long conversation history while keeping the current application state available. It also records activity and limits the work.

Watch how the agent updates its plan and uses the saved preference. Review the email before approving or declining it. Short runs may not need any history shortening.


In [ ]:
act4 = build_act4(
    scenario="event-shortage",
    model=MODEL,
    fixture=USE_FIXTURES,
    remember_preferences=("Prefer a concise supplier email.",),
    memory_path=MEMORY_PATH,
    output_root=OUTPUT_ROOT,
)

with Recorder(DB_PATH) as recorder:
    act4_result = await act4.run(
        recorder,
        reviewer=review_email,
        progress=print,
    )

print_harness_result(act4_result)
print("\n🧠 Saved preferences:", act4_result["preferences"])
print("📝 Final task list:")
for task in act4_result["tasks"]:
    print(task)


### View the recorded conversation and checks

The left panel shows the recorded task, the final assistant answer when available, and application results. The right panel shows checks, review decisions, and expandable evidence. Internal model replies remain in the evidence panel.

This cell reads the saved run; it does not call the model or repeat any action. Using `DB_PATH` works after the run cell closes its recorder. Add `backstage=False` to show only the request, response, and application results.


In [ ]:
show_chat(DB_PATH, act4_result["run_id"])


### Check the lesson automatically

From the repository terminal, run `python -m tests -p test_notebook.py` to execute the saved lesson cells with local scripted responses. The tests skip cloning, package installation, and credential entry; they use temporary output files and simulate review decisions. Run `python -m tests` for the full offline suite.
